## gCRL-VAE applied on the Lee dataset

- Load Lee adata, formatted for gCRL-VAE (adata.X is norm. logp1 transformed, important info added to adata.var and adata.obs)
- Compute eigengenes
- Train gCRL-VAE
- Validate

In [1]:
# ensuring all packages are reloaded each time I run a cell
%load_ext autoreload
%autoreload 2

In [2]:
# importing
import numpy as np
import pandas as pd
import scanpy as sc
import pickle
from gcrl.training.train_gcrl_vae import VAEConfig
from gcrl.training.train_gcrl_vae import train_gcrl_vae
from gcrl.grn.eigengenes import compute_eigengenes
from gcrl.evaluation import evaluate_gcrl_vae, EvalConfig

In [3]:
# loading data
adata = sc.read_h5ad("../../data/real/lee.h5ad")
adata

AnnData object with n_obs × n_vars = 3910 × 25970
    obs: 'cell_type', 'intervention', 'set', 'ct_int'
    var: 'kind', 'community'
    uns: 'cell_type_colors', 'gcrl_meta', 'intervention_colors', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [4]:
# keeping only interventions with more than 100 cells

# quantifying the number of cells for each intervention (conditioned to the cell type)
ct_int_cell_count = adata.obs['ct_int'].value_counts() 

# which intervention to retain?
to_retain = ct_int_cell_count[ct_int_cell_count > 100].index.to_list()
print(to_retain)

# selection
adata = adata[adata.obs['ct_int'].isin(to_retain)].copy()
print(adata)

['MAIT|unperturbed', 'γδT|unperturbed', 'MAIT|Rorc', 'NKT|Rorc', 'γδT|Rorc', 'NKT|unperturbed', 'NKT|Tbx21', 'MAIT|Tbx21']
AnnData object with n_obs × n_vars = 3834 × 25970
    obs: 'cell_type', 'intervention', 'set', 'ct_int'
    var: 'kind', 'community'
    uns: 'cell_type_colors', 'gcrl_meta', 'intervention_colors', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'


In [5]:
# setting interved NKT cells as test set
adata.obs['set'] = 'training'
adata.obs.loc[adata.obs['ct_int'] == 'NKT|Tbx21', 'set'] = 'test'
adata.obs.loc[adata.obs['ct_int'] == 'NKT|Rorc', 'set'] = 'test'
# pd.crosstab(index = adata.obs['set'], columns = [adata.obs['cell_type'], adata.obs['intervention']])

In [6]:
# keeping only highly variable genes (hvg) and TFs

# computing hvg
sc.pp.highly_variable_genes(adata)

# selection
adata = adata[:, ((adata.var['highly_variable']) | (adata.var['kind'] == 'TF'))].copy()

In [7]:
# computing eigengenes
compute_eigengenes(adata)
adata

AnnData object with n_obs × n_vars = 3834 × 4293
    obs: 'cell_type', 'intervention', 'set', 'ct_int'
    var: 'kind', 'community', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'cell_type_colors', 'gcrl_meta', 'intervention_colors', 'neighbors', 'pca', 'umap', 'hvg', 'X_comm_eig_comm_ids', 'X_comm_eig_global_index', 'comm_eig_meta'
    obsm: 'X_pca', 'X_umap', 'X_comm_eig'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [8]:
# gCRL-VAE configuration
cfg = VAEConfig()
cfg.epochs = 1000
cfg.outdir = "../../results/real/Lee/VAE"
cfg

VAEConfig(outdir='../../results/real/Lee/VAE', intervention_mapping='hard', batch_size=256, epochs=1000, lr=0.002, beta_kld=0.001, alpha_rec=1.0, lambda_mcc=1.0, num_workers=0, seed=0)

In [9]:
%%capture captured_output
# training gCRL-AE
model, history = train_gcrl_vae(adata=adata, cfg = cfg)
#adata = sc.read_h5ad(cfg.outdir + '/' + 'lee_post_training.h5ad')
#file_path = cfg.outdir + '/' + 'lee_model.pkl'
#with open(file_path, 'rb') as file:
#        model = pickle.load(file)

In [10]:
# saving
adata.write_h5ad(cfg.outdir + '/' + 'lee_post_training.h5ad')
with open(cfg.outdir + '/' + 'lee_model.pkl', 'wb') as file:
    pickle.dump(model, file)
with open(cfg.outdir + '/' + 'lee_history.pkl', 'wb') as file:
    pickle.dump(model, file)

In [11]:
eval_cfg = EvalConfig(
    set_key="set",                 # must match adata.obs column
    intervention_key="intervention",
    cell_type_key="cell_type",
    control_label="unperturbed",   # must match your control token
    out_dir=cfg.outdir,  # where CSVs/plots go
    make_umap_plots=True,          # turn off for speed if you like
    umap_neighbors=20, umap_min_dist=0.2, umap_metric="cosine",
    mu_all_per_ct=True,            # Diversity-style null per cell type
    topk=50,                      # DEGs used in “top-k” metrics
    knn_k_overlap=10
)

In [13]:
res = evaluate_gcrl_vae(adata, model=model, cfg=eval_cfg)

/home/laganiv/miniconda3/envs/deep_learning/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
